# Simple real NISAR GUNW workflow

Set `SNOWIN_GUNW` to a local NISAR GUNW product and optionally set `SNOWIN_NISAR_DEM` to a local NISAR-modified Copernicus DEM. The notebook inspects the normalized phase and all opened GUNW layers, then explicitly adds local-incidence geometry and previews pairwise dSWE. The correction layers are visualized for inspection and are not applied automatically.

In [ ]:
import os
from html import escape
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

from snowin import compute_dswe
from snowin.io import add_gunw_incidence, open_gunw
from snowin.io.gunw import read_attrs_hdf5, read_scalar_hdf5

gunw_text = os.environ.get("SNOWIN_GUNW")
if not gunw_text:
    gunw_text = input("Enter the path to a local NISAR GUNW product: ").strip()
if not gunw_text:
    raise RuntimeError("A local NISAR GUNW product path is required.")
GUNW_PATH = Path(gunw_text).expanduser().resolve()
DEM_PATH = os.environ.get("SNOWIN_NISAR_DEM")
DEM_CACHE = Path(os.environ.get("SNOWIN_DEM_CACHE", "~/.cache/snowin/dem")).expanduser()
def _format_metadata_value(value):
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.6g}"
    return str(value)

def _display_metadata(title, attrs, fields):
    rows = "".join(
        f"<tr><th><code>{escape(field)}</code></th><td>{escape(_format_metadata_value(attrs.get(field, 'not present')))}</td></tr>"
        for field in fields
    )
    display(HTML(f"<h3>{escape(title)}</h3><table><thead><tr><th>Field</th><th>Value</th></tr></thead><tbody>{rows}</tbody></table>"))

display(HTML(
    f"<h3>Inputs</h3><ul><li><b>GUNW product</b>: <code>{escape(str(GUNW_PATH))}</code></li><li><b>NISAR DEM</b>: <code>{escape(str(DEM_PATH or 'automatic download/cache'))}</code></li></ul>"
))

## Inspect product metadata

In [ ]:
phase_path = (
    "science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/HH/unwrappedPhase"
)
height_path = "science/LSAR/GUNW/metadata/radarGrid/heightAboveEllipsoid"
reference_path = "science/LSAR/identification/referenceZeroDopplerStartTime"
secondary_path = "science/LSAR/identification/secondaryZeroDopplerStartTime"

phase_attrs = read_attrs_hdf5(GUNW_PATH, phase_path)
height_attrs = read_attrs_hdf5(GUNW_PATH, height_path)
_display_metadata("Phase layer metadata", phase_attrs, ["units", "description", "grid_mapping", "min_value", "max_value", "mean_value", "sample_stddev"])
_display_metadata("Radar-grid height metadata", height_attrs, ["standard_name", "description", "units"])
reference_time = read_scalar_hdf5(GUNW_PATH, reference_path)
secondary_time = read_scalar_hdf5(GUNW_PATH, secondary_path)
display(HTML(f"<h3>Acquisition times</h3><ul><li><b>Reference</b>: <code>{escape(str(reference_time))}</code></li><li><b>Secondary</b>: <code>{escape(str(secondary_time))}</code></li></ul>"))

## Open and inspect the GUNW with SnowIn

This step does not download a DEM or compute incidence. Phase/product arrays remain lazy when Dask is installed, so you can inspect metadata before starting the slower geometry step.

In [ ]:
pair = open_gunw(GUNW_PATH, chunks="auto")
display(pair)
display(HTML(
    f"<p><b>Phase transform</b>: <code>{pair.attrs.get('phase_transform')}</code><br><b>Incidence status</b>: <code>{pair.attrs.get('incidence_angle_status')}</code><br><b>Wavelength</b>: <code>{pair.attrs.get('wavelength_m'):.9f} m</code></p>"
))

## Add local incidence when ready

This explicit call requires the GUNW path and appends `incidence_angle` and `geometry_valid` to the already-open dataset. It is the step that may download/reproject the NISAR DEM and read the GUNW LOS vectors.

In [ ]:
incidence_kwargs = {"dem_cache_dir": DEM_CACHE, "progress": True}
if DEM_PATH:
    incidence_kwargs["nisar_cop30_dem"] = Path(DEM_PATH).expanduser()
add_gunw_incidence(pair, GUNW_PATH, **incidence_kwargs)
display(pair[["incidence_angle", "geometry_valid"]])
geometry_info = {
    key: pair.attrs.get(key)
    for key in ("incidence_angle_source", "incidence_angle_reference", "dem_source", "dem_height_reference", "geometry_execution")
}
display(HTML(f"<h3>Geometry provenance</h3><pre>{escape(str(geometry_info))}</pre>"))

## Preview the opened layers, local incidence, and pairwise dSWE

This plot includes every layer returned by `open_gunw`, including available correction screens, plus the computed incidence and dSWE. Local incidence is displayed in degrees; SnowIn calculations use radians.

In [ ]:
step = max(1, min(pair.sizes["y"], pair.sizes["x"]) // 600)
safe_incidence = pair["incidence_angle"].where(pair["geometry_valid"])
dswe = compute_dswe(
    pair["phase"],
    safe_incidence,
    wavelength_m=float(pair.attrs["wavelength_m"]),
)
display(dswe)
phase_preview = pair["phase"].isel(y=slice(None, None, step), x=slice(None, None, step))
dswe_preview = dswe.isel(y=slice(None, None, step), x=slice(None, None, step))

plot_layers = list(pair.data_vars) + ["dswe"]
ncols = 3
nrows = max(1, (len(plot_layers) + ncols - 1) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.8 * nrows), squeeze=False)

for ax, name in zip(axes.flat, plot_layers):
    layer = dswe if name == "dswe" else pair[name]
    title = name.replace("_", " ").title()
    if "radar_height" in layer.dims:
        layer = layer.sel(radar_height=0, method="nearest")
        title += f" (radar height {float(layer.radar_height.values):g} m)"

    y_dim = "radar_y" if "radar_y" in layer.dims else "y"
    x_dim = "radar_x" if "radar_x" in layer.dims else "x"
    if name == "connected_component" or name == "geometry_valid":
        preview = layer.isel({y_dim: slice(None, None, step), x_dim: slice(None, None, step)})
        cmap = "tab20" if name == "connected_component" else "Greens"
    else:
        preview = layer.coarsen({y_dim: step, x_dim: step}, boundary="trim").mean()
        cmap = {
            "phase": "RdBu_r",
            "coherence": "plasma",
            "incidence_angle": "viridis",
            "dswe": "BrBG",
            "ionosphere": "RdBu_r",
            "ionosphere_unc": "magma",
            "hydro_tropo": "RdBu_r",
            "wet_tropo": "RdBu_r",
        }.get(name, "magma")
    kwargs = {"vmin": 0, "vmax": 1} if name == "geometry_valid" else {"robust": True}
    preview.compute().plot.imshow(ax=ax, cmap=cmap, **kwargs)
    ax.set_title(title)

for ax in axes.flat[len(plot_layers):]:
    fig.delaxes(ax)

fig.suptitle("Opened GUNW layers, local incidence, and pairwise dSWE")
fig.tight_layout()
plt.show()

In [ ]:
display(HTML(
f"<h3>Output checks</h3><table><tr><th>Check</th><th>Value</th></tr><tr><td>Phase definition</td><td><code>{pair.attrs.get("phase_difference_definition")}</code></td></tr><tr><td>Incidence reference</td><td><code>{pair["incidence_angle"].attrs.get("incidence_angle_reference")}</code></td></tr><tr><td>Geometry support fraction</td><td>{float(pair.geometry_valid.mean()):.4f}</td></tr><tr><td>Correction layers applied</td><td>{pair.attrs.get("correction_layers_applied", False)}</td></tr><tr><td>Finite phase fraction</td><td>{float(np.isfinite(phase_preview).mean()):.4f}</td></tr><tr><td>Finite dSWE fraction</td><td>{float(np.isfinite(dswe_preview).mean()):.4f}</td></tr></table>"
))
pair.close()